In [0]:
sql_server   = dbutils.secrets.get(scope="retailpulse-scope", key="sql-server-vault")
sql_database = dbutils.secrets.get(scope="retailpulse-scope", key="db-name")
sql_username = dbutils.secrets.get(scope="retailpulse-scope", key="retailpulse-sql-username")
sql_password = dbutils.secrets.get(scope="retailpulse-scope", key="retailpulse-sql-password")


STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope ="retailpulse-scope", key= "retailpulse-account-key")
STORAGE_ACCOUNT_NAME = "retailpulsestorage" 
CONTAINER_NAME       = "raw-data"


spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.blob.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

In [0]:
mount_path = dbutils.fs.ls(f"wasbs://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.blob.core.windows.net")
path = mount_path[1]

In [0]:
jdbc_url = (
    f"jdbc:sqlserver://{sql_server}:1433;"
    f"database={sql_database};"
    f"encrypt=true;"
    f"trustServerCertificate=false;"
    f"hostNameInCertificate=*.database.windows.net;"
    f"loginTimeout=30"
)

connection_properties = {
    "user":     sql_username,
    "password": sql_password,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Load Silver layer
df_clean = spark.read.parquet(f"{path.path}/clean_transactions")

print(f"Silver rows loaded: {df_clean.count():,}")

In [0]:
from pyspark.sql.functions import (
    col, to_date, year, month, dayofmonth,
    sum as spark_sum, count, countDistinct,
    round as spark_round
)

agg_daily_sales = df_clean \
    .withColumn("date", to_date(col("InvoiceDate"))) \
    .withColumn("date_key",
        (year("date") * 10000 +
         month("date") * 100 +
         dayofmonth("date")).cast("int")
    ) \
    .groupBy("date_key", "Country") \
    .agg(
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        countDistinct("Invoice").alias("total_orders"),
        spark_sum("Quantity").alias("total_quantity")
    ) \
    .withColumn(
        "avg_order_value",
        spark_round(col("total_revenue") / col("total_orders"), 2)
    ) \
    .withColumnRenamed("Country", "country") \
    .orderBy("date_key")

print(f"agg_daily_sales rows: {agg_daily_sales.count():,}")
display(agg_daily_sales.limit(5))

In [0]:
from pyspark.sql.functions import (
    col, max as spark_max, min as spark_min,
    sum as spark_sum, count, countDistinct,
    round as spark_round, datediff, lit,
    percent_rank, when, to_date
)
from pyspark.sql.window import Window

max_date = df_clean.agg(spark_max(to_date("InvoiceDate"))).collect()[0][0]
print(f"Reference date: {max_date}")

customer_metrics = df_clean \
    .groupBy("Customer ID") \
    .agg(
        spark_round(spark_sum("total_amount"), 2).alias("total_spent"),
        countDistinct("Invoice").alias("total_orders"),
        spark_sum("Quantity").alias("total_items"),
        spark_round(
            spark_sum("total_amount") / countDistinct("Invoice"), 2
        ).alias("avg_order_value"),
        spark_min(to_date("InvoiceDate")).alias("first_purchase_date"),
        spark_max(to_date("InvoiceDate")).alias("last_purchase_date")
    ) \
    .withColumn(
        "days_since_last",
        datediff(lit(max_date), col("last_purchase_date"))
    ) \
    .withColumnRenamed("Customer ID", "customer_id")

window = Window.orderBy("days_since_last")
window_asc = Window.orderBy(col("total_orders").asc())
window_mon = Window.orderBy(col("total_spent").asc())

customer_metrics = customer_metrics \
    .withColumn("recency_score",
        when(percent_rank().over(Window.orderBy("days_since_last")) <= 0.2, 5)
        .when(percent_rank().over(Window.orderBy("days_since_last")) <= 0.4, 4)
        .when(percent_rank().over(Window.orderBy("days_since_last")) <= 0.6, 3)
        .when(percent_rank().over(Window.orderBy("days_since_last")) <= 0.8, 2)
        .otherwise(1)
    ) \
    .withColumn("frequency_score",
        when(percent_rank().over(Window.orderBy("total_orders")) <= 0.2, 1)
        .when(percent_rank().over(Window.orderBy("total_orders")) <= 0.4, 2)
        .when(percent_rank().over(Window.orderBy("total_orders")) <= 0.6, 3)
        .when(percent_rank().over(Window.orderBy("total_orders")) <= 0.8, 4)
        .otherwise(5)
    ) \
    .withColumn("monetary_score",
        when(percent_rank().over(Window.orderBy("total_spent")) <= 0.2, 1)
        .when(percent_rank().over(Window.orderBy("total_spent")) <= 0.4, 2)
        .when(percent_rank().over(Window.orderBy("total_spent")) <= 0.6, 3)
        .when(percent_rank().over(Window.orderBy("total_spent")) <= 0.8, 4)
        .otherwise(5)
    )

customer_metrics = customer_metrics \
    .withColumn("rfm_segment",
        when((col("recency_score") >= 4) & (col("frequency_score") >= 4), "Champions")
        .when((col("recency_score") >= 3) & (col("frequency_score") >= 3), "Loyal Customers")
        .when((col("recency_score") >= 4) & (col("frequency_score") <= 2), "Recent Customers")
        .when((col("recency_score") <= 2) & (col("frequency_score") >= 4), "At Risk")
        .when((col("recency_score") <= 2) & (col("frequency_score") <= 2), "Lost")
        .otherwise("Potential Loyalists")
    )

print(f"Customer metrics rows: {customer_metrics.count():,}")
print("\nRFM Segment distribution:")
display(
    customer_metrics.groupBy("rfm_segment")
                    .count()
                    .orderBy(col("count").desc())
)

In [0]:
from pyspark.sql.functions import col

def execute_ddl(statement):
    conn = (spark._jvm.java.sql.DriverManager.getConnection(
        jdbc_url, sql_username, sql_password
    ))
    stmt = conn.createStatement()
    stmt.execute(statement)
    stmt.close()
    conn.close()

dim_customers = spark.read.jdbc(
    url=jdbc_url,
    table="dim_customers",
    properties=connection_properties
).select("customer_key", "customer_id")

agg_customer_final = customer_metrics.join(
    dim_customers, on="customer_id", how="left"
).select(
    "customer_key",
    "total_spent",
    "total_orders",
    "total_items",
    "avg_order_value",
    "first_purchase_date",
    "last_purchase_date",
    "days_since_last",
    "recency_score",
    "frequency_score",
    "monetary_score",
    "rfm_segment"
)

execute_ddl("TRUNCATE TABLE agg_daily_sales")
agg_daily_sales.write \
    .format("jdbc") \
    .option("url",      jdbc_url) \
    .option("dbtable",  "agg_daily_sales") \
    .option("user",     sql_username) \
    .option("password", sql_password) \
    .option("driver",   "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("batchsize", "5000") \
    .mode("append") \
    .save()
print("agg_daily_sales written")

execute_ddl("TRUNCATE TABLE agg_customer_metrics")
agg_customer_final.write \
    .format("jdbc") \
    .option("url",      jdbc_url) \
    .option("dbtable",  "agg_customer_metrics") \
    .option("user",     sql_username) \
    .option("password", sql_password) \
    .option("driver",   "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("batchsize", "5000") \
    .mode("append") \
    .save()
print("agg_customer_metrics written")

In [0]:
for table in ["agg_daily_sales", "agg_customer_metrics"]:
    df = spark.read \
        .format("jdbc") \
        .option("url",   jdbc_url) \
        .option("query", f"SELECT COUNT(*) as cnt FROM {table}") \
        .option("user",     sql_username) \
        .option("password", sql_password) \
        .option("driver",   "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()
    count = df.collect()[0]["cnt"]
    print(f" {table}: {count:,} rows")